# MasterAI: sual generasiyası üçün QLoRA fine-tune (gün 16)

Kiçik açıq modeli (defolt **Qwen2.5-3B-Instruct**) MasterAI-nin öz yaxşı sualları üzərində
**QLoRA** ilə öyrədirik: baza model 4-bit (NF4) yüklənir və dondurulur, yalnız kiçik LoRA
adapterləri (~0.5% parametr) öyrədilir. Pulsuz Colab T4 (16 GB) bəs edir.

**Addımlar:** dataset → 4-bit model → LoRA → öyrətmə → baza ilə müqayisə → birləşdirib saxla (gün 17: vLLM).

Dataset serverdə hazırlanır (tələbə məlumatı yoxdur, yalnız material və suallar):
```bash
cd /opt/masterai && .venv/bin/python -m ml.sft_dataset --data-dir data --out /root/sft
```
`train.jsonl` və `val.jsonl`-u bu notebook-a yüklə. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Paketlər
Versiyalar sabitlənib: TRL/PEFT API-ləri tez dəyişir, notebook bir ay sonra da işləməlidir.

In [ ]:
!pip -q install "transformers==4.46.3" "peft==0.13.2" "trl==0.12.2" "bitsandbytes==0.45.0" \
    "datasets==3.1.0" "accelerate==1.1.1"
!git clone -q https://github.com/Alihuseyn-ibrahimov/masterai.git
!pip -q install -r masterai/requirements.txt

## 2. Dataset

In [ ]:
import json, os
from google.colab import files

if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl")):
    files.upload()  # train.jsonl və val.jsonl seç

def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows, val_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl")
print(len(train_rows), "train,", len(val_rows), "val nümunəsi")
print(train_rows[0]["messages"][1]["content"][:400])

## 3. Baza model 4-bit (QLoRA)
NF4 + double quantization: 3B model ~2.5 GB VRAM. 7B üçün `BASE_MODEL`-i dəyiş (T4-də də sığır, amma yavaşdır).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "masterai-mcq-qlora"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 bf16 dəstəkləmir
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## 4. Öyrətmə
Nümunələr chat formatındadır (`messages`): istifadəçi mesajı produksiyadakı promptla bayt-bayt eynidir, cavab produksiya parserinin JSON formatıdır.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
val_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_rows])

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,      # effektiv batch 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=3072,                # 6000 simvolluq parça ~2000 token + cavab
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)
trainer = SFTTrainer(model=model, args=config, train_dataset=train_ds, eval_dataset=val_ds,
                     peft_config=lora, processing_class=tokenizer)
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model(OUTPUT_DIR)   # yalnız adapter (~50-100 MB)

## 5. Baza model vs fine-tune
Eyni val nümunələrində hər iki model sual yazır, `ml/score_generation.py` müqayisə edir: JSON oxunurmu, say düzgündürmü, qaydalı filtr, uzunluqla özünü verən cavab, materiala bağlılıq.

In [ ]:
model = trainer.model
model.eval()

def generate(messages, max_new_tokens=1500):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run(path, use_adapter):
    with open(path, "w", encoding="utf-8") as fh:
        for row in val_rows:
            if use_adapter:
                text = generate(row["messages"])
            else:
                with model.disable_adapter():
                    text = generate(row["messages"])
            fh.write(json.dumps({"output": text}, ensure_ascii=False) + "\n")

run("base.jsonl", use_adapter=False)
run("qlora.jsonl", use_adapter=True)

In [ ]:
!cd masterai && python -m ml.score_generation --val ../val.jsonl --pred ../base.jsonl --pred ../qlora.jsonl

## 6. Birləşdir və saxla (gün 17: vLLM)
vLLM adapteri ayrıca da yükləyə bilir, amma birləşdirilmiş fp16 model sadədir. Hugging Face-ə yükləmək üçün `HF_TOKEN` Colab Secrets-də olmalıdır.

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained("masterai-mcq-merged", safe_serialization=True)
tokenizer.save_pretrained("masterai-mcq-merged")

# İstəsən: merged.push_to_hub("<hf-istifadəçi>/masterai-mcq-3b", private=True)
#          tokenizer.push_to_hub("<hf-istifadəçi>/masterai-mcq-3b", private=True)
!du -sh {OUTPUT_DIR} masterai-mcq-merged